# 07 | Power BI Export Validation & Model Readiness

**Project:** Dubai Short-Term Rental Intelligence  
**Data provider:** AirROI Dubai (third-party Kaggle release; not Airbnb internal data)  
**Reporting period:** March 2021–February 2026  
**Purpose:** Verify that the six analytical exports faithfully reproduce the validated SQLite database, preserve source measures, and support the intended Power BI model.

**Decision gate:** The files are approved for **Power BI import** only if the checks below pass. Approval does not mean that an interactive Power BI report has been built or validated. Reported occupancy and revenue are **provider estimates** for the observed sample, not verified bookings or Dubai-wide market totals.

## 1. Execution context and scope

Run this notebook after the preparation, benchmarking, and database-build notebooks (03–05) have completed. The source transformations are maintained in `src/01_validate_and_prepare.py`, `src/02_build_peer_benchmarks.py` and `src/03_build_powerbi_exports.py`. The SQL database is the *independent comparison target* for the CSV exports; row counts alone are not considered proof of equivalence.

Notebook execution supports Jupyter launched from the repository root or `notebooks/`. Identifiers are loaded explicitly as text to protect long Airbnb listing IDs. No raw data is altered.

**Snapshot control (independent audit, 10 October 2026):** Fixed figures in this narrative refer to the four original files identified by SHA-256 in `reports/source_audit.json`. Executable tables derive values at runtime. Review the narrative if replacing the source snapshot. Raw data and listing-level outputs remain private; conclusions describe this sample and provider estimates.

In [1]:
from pathlib import Path
import json
import sqlite3
import numpy as np
import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (CWD, *CWD.parents)
                     if (p / "src/03_build_powerbi_exports.py").exists()
                     and (p / "data/processed").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the project root or notebooks/ directory.")
DB_PATH = PROJECT_ROOT / "data/processed/dubai_rentals.sqlite"
EXPORTS = PROJECT_ROOT / "powerbi/data"
REPORTS = PROJECT_ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
if not DB_PATH.exists():
    raise FileNotFoundError("Database missing. Run Notebook 05 with the updated builder first.")

TABLES = (
    "dim_listing", "dim_month", "fact_listing_month",
    "fact_coverage_month", "fact_peer_benchmark", "peer_examples_latest",
)
KEYS = {
    "dim_listing": ["listing_id"], "dim_month": ["month_date"],
    "fact_listing_month": ["listing_id", "month_date"],
    "fact_coverage_month": ["month_date"],
    "fact_peer_benchmark": ["listing_id", "month_date"],
    "peer_examples_latest": ["listing_id", "month_date", "peer_listing_id"],
}
frames = {
    t: pd.read_csv(EXPORTS / f"{t}.csv",
                   dtype={"listing_id": "string", "peer_listing_id": "string", 
                          "host_id": "string", "property_type": "string", "superhost": "string"},
                   low_memory=False)
    for t in TABLES
}
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys=ON")
results = []
def check(test, passed, detail):
    results.append({"validation": test, "result": "PASS" if bool(passed) else "FAIL", "evidence": str(detail)})

print(f"Project: {PROJECT_ROOT.name} | SQLite: {DB_PATH.name}")
display(pd.DataFrame([{"export": t + ".csv", "rows": len(frames[t]),
                       "columns": len(frames[t].columns)} for t in TABLES]))

## 2. Source-to-database schema and row-level reconciliation

For every export, compare the column contract and every data cell against the current SQLite table. Numerical columns are compared with a small floating-point tolerance; identifiers and categorical values must agree exactly. Also confirm that listing and host identifiers use SQLite **TEXT**, not `REAL` (which can misrepresent identifiers and mislead Power BI type inference).

In [2]:
schema = {}
for table in TABLES:
    info = pd.read_sql_query(f"PRAGMA table_info({table})", conn)
    schema[table] = dict(zip(info["name"], info["type"]))
    df = frames[table]
    expected_columns = list(schema[table])
    check(f"{table}: column schema", list(df.columns) == expected_columns,
          f"{len(df.columns)} exported / {len(expected_columns)} expected columns")
    db_df = pd.read_sql_query(f"SELECT * FROM {table}", conn,
                              dtype={c: "string" for c in ["listing_id", "peer_listing_id", "host_id"]
                                     if c in schema[table]})
    check(f"{table}: database row count", len(df) == len(db_df),
          f"CSV={len(df):,}; DB={len(db_df):,}")
    if list(df.columns) != list(db_df.columns) or len(df) != len(db_df):
        check(f"{table}: row-level values", False, "Cannot compare with schema/row-count mismatch")
        continue
    df = df.sort_values(KEYS[table]).reset_index(drop=True)
    db_df = db_df.sort_values(KEYS[table]).reset_index(drop=True)
    ok = True
    for column, kind in schema[table].items():
        a, b = df[column], db_df[column]
        if "TEXT" in kind.upper():
            equal = a.astype("string").fillna("<NULL>").eq(
                b.astype("string").fillna("<NULL>")).all()
        else:
            aa, bb = pd.to_numeric(a, errors="coerce"), pd.to_numeric(b, errors="coerce")
            equal = np.isclose(aa.to_numpy(dtype=float), bb.to_numpy(dtype=float),
                               rtol=1e-10, atol=1e-6, equal_nan=True).all()
        if not equal:
            ok = False
            print(f"Mismatch: {table}.{column}")
    check(f"{table}: row-level values", ok, "All exported values match SQLite" if ok else "Value mismatch")

for column in ("host_id", "property_type", "superhost", "listing_id"):
    check(f"dim_listing.{column}: SQLite TEXT", schema["dim_listing"].get(column) == "TEXT",
          f"SQLite type={schema['dim_listing'].get(column)}")
check("database integrity", conn.execute("PRAGMA integrity_check").fetchone()[0] == "ok", "PRAGMA integrity_check")
check("foreign keys enabled", conn.execute("PRAGMA foreign_keys").fetchone()[0] == 1, "PRAGMA foreign_keys")
violations = conn.execute("PRAGMA foreign_key_check").fetchall()
check("foreign key consistency", len(violations) == 0, f"{len(violations)} violations")

display(pd.DataFrame(results).tail(12))

## 3. Fact grains, relationship keys, and domain constraints

The model uses shared listing and month dimensions with two fact tables; direct fact-to-fact Power BI joins would introduce ambiguous filtering. The competitor-examples table is a **latest-month, disconnected explanation table**. Its two listing IDs must both refer to valid fact rows for the same month. The detailed peer table intentionally contains *up to five examples per benchmark*, not every peer.

In [3]:
f, b = frames["fact_listing_month"], frames["fact_peer_benchmark"]
d, m = frames["dim_listing"], frames["dim_month"]
cov, examples = frames["fact_coverage_month"], frames["peer_examples_latest"]

for table in TABLES:
    df = frames[table]
    key = KEYS[table]
    null_or_blank = df[key].isna().any().any()
    for col in key:
        if col in ("listing_id", "peer_listing_id"):
            null_or_blank |= df[col].astype("string").str.strip().eq("").any()
    check(f"{table}: non-null keys", not null_or_blank, f"Keys: {', '.join(key)}")
    check(f"{table}: unique grain", not df.duplicated(key).any(), f"{len(df):,} records")

fkeys = pd.MultiIndex.from_frame(f[["listing_id", "month_date"]])
bkeys = pd.MultiIndex.from_frame(b[["listing_id", "month_date"]])
check("benchmark and fact: identical composite keys",
      len(f) == len(b) and fkeys.isin(bkeys).all() and bkeys.isin(fkeys).all(),
      f"{len(f):,} fact and {len(b):,} benchmark rows")
check("fact listing and month references", f.listing_id.isin(d.listing_id).all()
      and f.month_date.isin(m.month_date).all(), "Both dimension relationships resolved")
check("benchmark listing/month references", b.listing_id.isin(d.listing_id).all()
      and b.month_date.isin(m.month_date).all(), "Both dimension relationships resolved")
check("coverage month references", cov.month_date.isin(m.month_date).all(),
      f"{len(cov)} coverage months")
check("examples: both ends exist in fact",
      pd.MultiIndex.from_frame(examples[["listing_id", "month_date"]]).isin(fkeys).all()
      and pd.MultiIndex.from_frame(examples[["peer_listing_id", "month_date"]].rename(
          columns={"peer_listing_id": "listing_id"})).isin(fkeys).all(),
      f"{len(examples):,} peer-detail records")
check("examples: no self-comparisons", examples.listing_id.ne(examples.peer_listing_id).all(),
      "Subject != competitor")

latest = f.month_date.max()
check("examples: latest month only", examples.month_date.eq(latest).all(),
      f"Latest observed month: {latest}")
check("all month keys parse as month-start dates", all(
    pd.to_datetime(frames[t]["month_date"], format="%Y-%m-%d", errors="raise").dt.day.eq(1).all()
    for t in TABLES if "month_date" in frames[t].columns), "YYYY-MM-01 dates")
check("listing identifiers are strings", all(
    pd.api.types.is_string_dtype(frames[t][key].dtype)
    for t in TABLES for key in ("listing_id", "peer_listing_id")
    if key in frames[t].columns),
    f"Longest listing ID: {f.listing_id.str.len().max()} characters")
check("active flags are binary", f.active.notna().all()
      and f.active.isin([0, 1]).all(), "Expected 0 / 1")
check("provider occupancy is 0–1", f.occupancy.dropna().between(0,1).all(),
      "Formatted as percentage in Power BI")
check("native AED measures are non-negative", all(
    f[column].dropna().ge(0).all()
    for column in ("native_avg_daily_rate", "native_revenue", "native_revpar")),
    "Nonnegative provider measures")
check("all source rates use AED", f.currency.eq("AED").all(), "Currency label")
check("ineligible benchmarks suppress premiums",
      b.loc[b.comparison_status.ne("Eligible"), "premium_pct"].isna().all(),
      "No fabricated benchmarks for ineligible listings")
check("eligible benchmark minimum peer count", b.loc[b.comparison_status.eq("Eligible"),
      "peer_count"].ge(10).all(), "At least 10 comparable properties")
check("peer radii limited to 2 km or 5 km", b.radius_km.dropna().isin([2, 5]).all(),
      "Matching radius rule")
check("peer-detail distances within matched radius", examples.distance_km.ge(0).all()
      and examples.distance_km.le(5.001).all(), "Maximum fallback distance: 5 km")
display(pd.DataFrame(results).tail(14))

## 4. Independent monthly-coverage reconciliation — critical export fix

A previous cached export contained **60 missing values** in `same_listing_prev_month`. This metric is **a count of listings observed in the preceding calendar month**, not a Boolean flag and not an active-listing count. The updated source script generates it from monthly fact records; the updated database builder rejects blank or inconsistent values. The first month must equal zero. `coverage_change_pct` is allowed to be null for the first month because there is no prior observation.

We independently recalculate the monthly coverage figures from `fact_listing_month.csv` and compare every reporting period.

In [4]:
# Reconstruct continuity from dates rather than trusting the exported flag.
ordered_fact=f.sort_values(['listing_id','month_date']).copy()
previous=ordered_fact.groupby('listing_id').month_date.shift()
expected_previous=(pd.to_datetime(ordered_fact.month_date)-pd.offsets.MonthBegin(1)).dt.strftime('%Y-%m-%d')
independent_returning=previous.eq(expected_previous)
check('fact: prior-month observation flag independently verified',
      ordered_fact.same_listing_prev_month.eq(independent_returning).all(), 'Calendar-adjacent source dates')
f=f.copy()
f['same_listing_prev_month']=independent_returning.reindex(f.index)
monthly = f.groupby("month_date", as_index=False).agg(
    observed_listings=("listing_id", "size"),
    active_listings=("active", "sum"),
    same_listing_prev_month=("same_listing_prev_month", "sum"),
    neighborhood_missing_pct=("neighborhood", lambda s: s.isna().mean() * 100),
    bedrooms_missing_pct=("bedrooms", lambda s: s.isna().mean() * 100),
)
active = f[f.active.eq(1)]
medians = active.groupby("month_date", as_index=False).agg(
    median_active_adr_aed=("native_avg_daily_rate", "median"),
    median_active_estimated_occupancy=("occupancy", "median"),
)
monthly = monthly.merge(medians, on="month_date", how="left", validate="one_to_one")
monthly = monthly.sort_values("month_date").reset_index(drop=True)
monthly["coverage_change_pct"] = monthly.observed_listings.pct_change(fill_method=None) * 100
reconciled = cov.merge(monthly, on="month_date", how="outer", validate="one_to_one",
                       suffixes=("_export", "_derived"), indicator=True)
check("coverage: exactly same reporting months", reconciled._merge.eq("both").all()
      and len(reconciled) == len(monthly), f"{len(reconciled)} months")
for metric in ("observed_listings", "active_listings", "same_listing_prev_month",
               "neighborhood_missing_pct", "bedrooms_missing_pct", "median_active_adr_aed",
               "median_active_estimated_occupancy", "coverage_change_pct"):
    a, z = reconciled[f"{metric}_export"], reconciled[f"{metric}_derived"]
    valid = np.isclose(a.to_numpy(dtype=float), z.to_numpy(dtype=float),
                       rtol=1e-9, atol=1e-6, equal_nan=True).all()
    check(f"coverage: {metric} reconciles", valid, f"Recalculated for {len(reconciled)} months")
check("coverage: returning-listing count populated", cov.same_listing_prev_month.notna().all(),
      f"Non-null: {cov.same_listing_prev_month.notna().sum()}/{len(cov)}")
check("coverage: first returning count equals zero",
      int(cov.sort_values("month_date").iloc[0].same_listing_prev_month or 0) == 0,
      "First month cannot have prior-month returns")
check("coverage: returning count never exceeds observed",
      cov.same_listing_prev_month.le(cov.observed_listings).all(), "Physical upper bound")
print("Monthly continuity spot-check:")
display(cov[["month_date", "observed_listings", "active_listings",
             "same_listing_prev_month", "coverage_change_pct"]].head(5))

## 5. Independent latest-month KPI and review-queue checks

A Power BI headline KPI must use an explicit reporting month; a distinct listing count aggregated across the full history answers a different question. The checks below reconcile the latest month with SQL directly and verify that the pricing-review view captures only benchmarked listings exceeding the documented deviation thresholds.

In [5]:
latest_rows = f.loc[f.month_date.eq(latest)]
latest_active = latest_rows.loc[latest_rows.active.eq(1)]
latest_bench = b.loc[b.month_date.eq(latest)]
view_queue = pd.read_sql_query("SELECT listing_id FROM vw_latest_price_review_queue", conn,
                               dtype={"listing_id": "string"})
expected_queue = latest_bench.loc[
    latest_bench.comparison_status.eq("Eligible")
    & latest_bench.review_signal.isin(["Review", "Large deviation"]), "listing_id"
]
check("latest observed listings: SQL count", len(latest_rows) == conn.execute(
    "SELECT COUNT(*) FROM fact_listing_month WHERE month_date=?", (latest,)).fetchone()[0],
    f"{len(latest_rows):,} observed")
check("latest active listings: SQL count", len(latest_active) == conn.execute(
    "SELECT COUNT(*) FROM fact_listing_month WHERE month_date=? AND active=1", (latest,)).fetchone()[0],
    f"{len(latest_active):,} active")
check("latest benchmark eligibility: SQL count", latest_bench.comparison_status.eq("Eligible").sum()
      == conn.execute("SELECT COUNT(*) FROM fact_peer_benchmark WHERE month_date=? AND comparison_status='Eligible'",
                      (latest,)).fetchone()[0], "Eligible comparator count")
check("pricing review queue: exact property IDs", set(view_queue.listing_id) == set(expected_queue)
      and len(view_queue) == len(expected_queue), f"{len(view_queue):,} pricing-review listings")
check("monthly coverage: current reporting month agrees", len(latest_rows) ==
      int(cov.set_index("month_date").loc[latest, "observed_listings"]),
      "Current coverage count matches fact")

snapshot = pd.DataFrame([
    {"metric": "Latest reporting month", "observed result": latest},
    {"metric": "Observed listings", "observed result": f"{len(latest_rows):,}"},
    {"metric": "Active listings", "observed result": f"{len(latest_active):,}"},
    {"metric": "Median reported rate (active, AED)",
     "observed result": f"{latest_active.native_avg_daily_rate.median():,.0f}"},
    {"metric": "Eligible competitor benchmarks",
     "observed result": f"{int(latest_bench.comparison_status.eq('Eligible').sum()):,}"},
    {"metric": "Properties in pricing-review queue", "observed result": f"{len(view_queue):,}"},
])
display(snapshot)

## 6. Power BI relationship and formatting contracts

The recommended **single-direction** relationships are `dim_listing` → both fact tables by listing ID, `dim_month` → both fact tables by month, and `dim_month` → `fact_coverage_month` on month (one-to-one). The coverage table is **global**, and must not be displayed as if property slicers affect it. The latest-month competitor examples remain **disconnected**; use an explicit DAX selection measure rather than ambiguous dual listing relationships.

- **Text:** `listing_id`, `peer_listing_id`, `host_id`, `property_type`, `superhost` (when available).
- **Date:** `month_date` as a date at month-start grain; sort month labels using `month_sort`.
- **Rates/revenue:** AED decimal currency. `native_avg_daily_rate` is the source's reported daily rate.
- **Estimated occupancy:** fraction in the exported data (`0.22` → `22%` in Power BI).
- **Pricing premium:** percentage *points* in exported data (`25` means `25%`); format as `0.0"%"`, not percent format that multiplies by 100 again.
- **Historical listing dimension:** latest known attributes, not verified snapshots for older months.
- **Benchmark information:** `peer_count` and selected radius are required context for price comparisons. An eligible benchmark is a screening signal, not an optimal-price recommendation.

These relationship/filter contracts must be implemented and checked in Power BI itself after import; Python cannot prove report filter behavior.

In [6]:
data_types = pd.DataFrame([
    {"table": "All listing tables", "field": "listing_id, peer_listing_id", "Power BI type": "Text"},
    {"table": "dim_listing", "field": "host_id, property_type", "Power BI type": "Text"},
    {"table": "Time-aware tables", "field": "month_date", "Power BI type": "Date"},
    {"table": "fact_listing_month", "field": "active", "Power BI type": "True/False (CSV; SQLite uses 0/1)"},
    {"table": "fact_listing_month", "field": "occupancy", "Power BI type": "Decimal, percentage display"},
    {"table": "fact_listing_month", "field": "native_avg_daily_rate", "Power BI type": "Decimal, AED"},
    {"table": "fact_peer_benchmark", "field": "premium_pct", "Power BI type": 'Decimal, custom "0.0%" display'},
    {"table": "dim_listing and facts", "field": "bedrooms", "Power BI type": "Whole number / blank"},
])
display(data_types)

## 7. Data readiness summary and audit exports

The result is **conditional on every test passing**, rather than a manually written success message. The detailed audit is exported for reproducibility. The notebook reports missingness and source limitations but never silently imputes property details or estimates.

In [7]:
audit = pd.DataFrame(results)
summary = audit.groupby("result").size().to_dict()
display(audit)
print(f"Checks: {len(audit)} | Passed: {summary.get('PASS', 0)} | Failed: {summary.get('FAIL', 0)}")

audit_path = REPORTS / "powerbi_export_validation_v12.csv"
summary_path = REPORTS / "powerbi_data_readiness_v12.json"
audit.to_csv(audit_path, index=False)
readiness = {
    "source": "AirROI Dubai third-party dataset",
    "latest_reporting_month": latest,
    "sqlite_database": str(DB_PATH.relative_to(PROJECT_ROOT)),
    "files": {t: {"rows": len(frames[t]), "columns": len(frames[t].columns)} for t in TABLES},
    "validation": {"total": len(audit), "passed": summary.get("PASS", 0),
                   "failed": summary.get("FAIL", 0)},
    "powerbi_import_approved": summary.get("FAIL", 0) == 0,
    "interactive_powerbi_report_verified": False,
    "methodology_note": "Estimated occupancy and revenue are AirROI estimates; sample is not a full Dubai census",
}
summary_path.write_text(json.dumps(readiness, indent=2), encoding="utf-8")
print("Saved:", audit_path.relative_to(PROJECT_ROOT), "and", summary_path.relative_to(PROJECT_ROOT))
if summary.get("FAIL", 0):
    raise AssertionError("Power BI import NOT approved. Investigate failed validation rows above.")
print("PASS — The six export files are ready for Power BI import under the documented model contract.")
conn.close()

## Key Findings and Analytical Decisions

**Data readiness.** The six CSVs were tested against a rebuilt SQLite database with declared primary/foreign keys, row-level export comparisons, full listing-month grain validation, and monthly-coverage recalculation. Saved outputs were removed for GitHub. Authorized local execution regenerates the approval decision, check outcomes and private `reports/powerbi_export_validation_v12.csv`; historical evidence is summarized in [Validation Summary](../VALIDATION_SUMMARY.md).

**Resolved export defect.** The monthly returning-listing count had previously been blank in every record of a cached coverage export. Rebuilding the source tables and strengthening upstream validation resolved the defect; this notebook independently recalculates the count for every month and will fail if the issue recurs.

**Model implementation decisions.** Use shared listing/date dimensions, single-direction filters, a global monthly-coverage table, and a deliberately disconnected latest-month peer-examples table. Protect long IDs from numeric coercion. Apply month filters to property KPIs and treat `premium_pct` as percentage points.

**Business limitations.** Source occupancy/revenue remain provider estimates; metadata is sparse and latest-observed, and a peer price deviation is a review signal, not evidence of suboptimal prices. Validation of Python and CSV files does **not** establish that Power BI DAX, visuals, or drill-through interactions are correct. The six-page Power BI application is already completed; its existing native/user evidence is separate from this historical intermediate-data validation stage.

**Next step.** Notebook 08 documents the analytical brief. Run `src/08_export_model_imports.py --expect-reference` after intermediate preparation to produce the twelve imports for the completed [PBIP](../powerbi/DubaiRentalIntelligence.pbip). See [Local Setup](../docs/SETUP.md); no report/model regeneration is required.